# Hipótese 4: O detrator crônico

Testa se existe um **traço individual de propensão à detração**, ou seja, se o Cliente que
detratou uma vez tende a detratar de novo mesmo quando o voo seguinte não apresenta falha
operacional.

A hipótese é avaliada por três testes encadeados, que vão retirando progressivamente as
explicações operacionais alternativas:

1. **Concentração**: a repetição da detração ocorre mais do que o acaso explicaria?
2. **Predição**: o histórico separa grupos com risco diferente?
3. **Resistência ao controle**: a diferença sobrevive quando isolamos voos sem falha?

Ao final, um modelo de regressão logística estima o efeito do histórico já descontadas as
condições operacionais do voo.

Variável alvo: `DETRATOR`, definida como `NPS_PRINCIPAL == -100`.

**Base de entrada.** O notebook lê as quatro partições de NPS, a base de informação de viagem e,
se necessário, a de perfil do Cliente, e aplica a mesma regra de deduplicação de
`scripts/preprocessamento_nps.py`, chegando aos 484.915 registros da base analítica definida na
seção A.1.3. A pasta `data/` não é versionada, por compromisso entre o Inteli e o parceiro, então
os arquivos precisam existir localmente antes da execução. Cada seção numerada abaixo corresponde
a um ponto citado na seção 4.2.3 do documento, para que qualquer número publicado possa ser
rastreado até a célula que o produziu.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

pd.set_option('display.width', 150)

## 1. Carregar e unir as bases

In [ ]:
# Os arquivos podem estar em data/raw, que e o padrao do repositorio, no diretorio do notebook
# ou em /content, onde o Colab deposita o upload. Os nomes vem do parceiro com ponto
# (PROJETO_INTELI.NPS_01.csv) e costumam virar underscore depois do upload.
import glob, os

PASTAS = ['data/raw', '../data/raw', '/content/data/raw', 'data', '../data', '/content', '.']

def achar(padrao):
    for pasta in PASTAS:
        for nome in (f'PROJETO_INTELI.{padrao}.csv', f'PROJETO_INTELI_{padrao}.csv'):
            caminho = os.path.join(pasta, nome)
            if os.path.exists(caminho):
                return caminho
    encontrados = sorted({
        c for pasta in PASTAS for c in glob.glob(os.path.join(pasta, f'*{padrao}*.csv'))
    })
    if not encontrados:
        raise FileNotFoundError(f'Nao encontrei o arquivo de {padrao}. Procurei em {PASTAS}')
    if len(encontrados) > 1:
        # Pegar o primeiro em silencio ja escolheu arquivo errado antes, quando havia copia
        # numerada do tipo "PERFIL_CLIENTE_02 (1).csv" ao lado do original.
        raise ValueError(f'Mais de um arquivo candidato a {padrao}: {encontrados}')
    return encontrados[0]

nps = pd.concat(
    [pd.read_csv(achar(f'NPS_0{i}')) for i in range(1, 5)],
    ignore_index=True
)
viagem = pd.read_csv(achar('INFORMACAO_VIAGEM'))

# Um unico RESPONDENT_ID aparece duas vezes na concatenacao de NPS_01 a NPS_04, com dois valores
# de TEMPO_VOO e todo o resto identico. A regra da base analitica, em
# scripts/preprocessamento_nps.py, mantem a primeira ocorrencia e consolida TEMPO_VOO pela media.
# Como esta analise nao usa TEMPO_VOO, manter a primeira ocorrencia e equivalente.
duplicados = int(nps['RESPONDENT_ID'].duplicated().sum())
nps = nps.drop_duplicates(subset='RESPONDENT_ID', keep='first')

df = nps.merge(viagem, on='RESPONDENT_ID', how='inner')

# ID_GOLDENRECORD identifica o Cliente e permite ligar respostas da mesma pessoa.
if 'ID_GOLDENRECORD' not in df.columns:
    perfil = pd.concat(
        [pd.read_csv(achar(f'PERFIL_CLIENTE_0{i}')) for i in range(1, 3)],
        ignore_index=True
    )
    df = df.merge(perfil[['RESPONDENT_ID', 'ID_GOLDENRECORD']], on='RESPONDENT_ID', how='left')

# A base analitica documentada na secao A.1.3 tem 484.915 registros. Se este numero mudar, a
# entrada nao e a mesma e os valores publicados na secao 4.2.3 deixam de valer.
assert len(df) == 484_915, f'esperava 484.915 registros na base analitica, obtive {len(df)}'

# DATA_STD vem em ISO (aaaa-mm-dd) nas quatro particoes. O formato e declarado de proposito:
# deixar a inferencia com dayfirst=True troca dia por mes nesta coluna.
df['DATA'] = pd.to_datetime(df['DATA_STD'], format='%Y-%m-%d')

# CANCELAMENTO_VOO chega como booleano. Normalizar aqui, com verificacao, evita que um export em
# texto passe despercebido e esvazie em silencio os cortes B e C da secao 5.
if df['CANCELAMENTO_VOO'].dtype == bool:
    df['CANCELADO'] = df['CANCELAMENTO_VOO']
else:
    mapa = {'true': True, 'false': False, 'sim': True, 'nao': False, 'não': False}
    df['CANCELADO'] = df['CANCELAMENTO_VOO'].astype(str).str.strip().str.lower().map(mapa)
assert df['CANCELADO'].notna().all(), 'CANCELAMENTO_VOO tem valor fora do esperado'

df['DETRATOR'] = (df['NPS_PRINCIPAL'] == -100).astype(int)

print(f'RESPONDENT_ID duplicados removidos: {duplicados}')
print(f'Total de respostas: {len(df)}')
print(f'Clientes distintos: {df["ID_GOLDENRECORD"].nunique()}')
print(f'Taxa geral de detratores: {df["DETRATOR"].mean()*100:.1f}%')

## 2. Construir o histórico de detração

Para cada Cliente, as respostas são ordenadas **por data** e cada linha recebe o resultado da
**resposta anterior daquele mesmo Cliente**. Só entram na análise as respostas que têm uma anterior.

Este é o ponto que garante ausência de vazamento de informação: a resposta anterior já existia
antes do voo novo acontecer, então usá-la como preditor é legítimo. A ordenação é feita por
`DATA_STD` e não pela ordem do arquivo nem pelo `RESPONDENT_ID`, que acompanha a cronologia na
maior parte dos casos mas não em todos. Quando duas respostas do mesmo Cliente caem no mesmo dia,
o `RESPONDENT_ID` desempata de forma determinística, e a célula abaixo informa quantos pares
estão nessa situação.

In [ ]:
d = df.dropna(subset=['ID_GOLDENRECORD']).sort_values(
    ['ID_GOLDENRECORD', 'DATA', 'RESPONDENT_ID']
).copy()
g = d.groupby('ID_GOLDENRECORD')

d['DET_ANTERIOR']     = g['DETRATOR'].shift(1)
d['ATRASO_ANTERIOR']  = g['ATRASO_CHEGADA'].shift(1)
d['DATA_ANTERIOR']    = g['DATA'].shift(1)

hist = d.dropna(subset=['DET_ANTERIOR']).copy()

print(f'Respostas com histórico: {len(hist)}')
print(f'Cobertura sobre a base: {len(hist)/len(df)*100:.1f}%')
print(f'Pares no mesmo dia, desempatados por RESPONDENT_ID: {int((hist["DATA"] == hist["DATA_ANTERIOR"]).sum())}')

## 3. Teste 1: concentração

Entre os Clientes com **exatamente duas respostas**, se detratar fosse independente a cada
viagem, a combinação "detratou nas duas" deveria aparecer com a frequência esperada pelo
produto das marginais. O qui-quadrado mede o afastamento entre observado e esperado.

O valor publicado é o **sem correção de continuidade de Yates**. A célula imprime também o valor
com a correção, para deixar registrado que a escolha da convenção não muda a conclusão.

In [ ]:
contagem = d.groupby('ID_GOLDENRECORD')['DETRATOR'].size()
dois = d[d['ID_GOLDENRECORD'].isin(contagem[contagem == 2].index)]

par = dois.groupby('ID_GOLDENRECORD')['DETRATOR'].agg(['first', 'last'])
tabela = pd.crosstab(par['first'], par['last'])
tabela.index.name, tabela.columns.name = 'primeira resposta', 'segunda resposta'

chi2, p, gl, esperado = stats.chi2_contingency(tabela, correction=False)
chi2_yates, _, _, _ = stats.chi2_contingency(tabela, correction=True)

print(tabela, '\n')
print(f'Clientes com exatamente duas respostas: {int(tabela.values.sum())}')
print(f'"detratou nas duas": observado = {tabela.loc[1, 1]}  |  esperado = {esperado[1][1]:.0f}')
print(f'qui-quadrado = {chi2:.1f}   gl = {gl}   p = {p:.3e}')
print(f'qui-quadrado com correcao de Yates = {chi2_yates:.1f}')

## 4. Teste 2: predição

Entre as respostas que têm histórico, compara-se a taxa de detração de quem detratou antes
contra a de quem não detratou.

In [ ]:
taxa = hist.groupby('DET_ANTERIOR')['DETRATOR'].agg(['size', 'mean'])
taxa['mean'] = taxa['mean'] * 100
taxa.index = ['nao detratou antes', 'detratou antes']
taxa.columns = ['n', 'taxa de detracao (%)']

print(taxa.round(1))
print(f'\nRazao entre os grupos: {taxa.iloc[1, 1] / taxa.iloc[0, 1]:.1f}x')

## 5. Teste 3: resistência ao controle

Aqui está o teste decisivo. Se a repetição fosse apenas consequência de o Cliente ter pegado
voos piores, a diferença deveria **encolher** quando isolamos apenas voos sem falha. Os cortes
são progressivamente mais restritivos:

- **Corte A**: todas as respostas com histórico
- **Corte B**: voo atual sem atraso na saída, sem atraso na chegada e sem cancelamento
- **Corte C**: o corte B, exigindo também que o voo anterior tenha chegado sem atraso

In [ ]:
voo_perfeito = (
    (hist['ATRASO_CHEGADA'] == 0)
    & (hist['ESTATISTICA_ATRASOSAIDA'] == 0)
    & (~hist['CANCELADO'])
)

cortes = {
    'A. todas as respostas com historico': hist,
    'B. voo atual sem falha':              hist[voo_perfeito],
    'C. B + voo anterior sem atraso':      hist[voo_perfeito & (hist['ATRASO_ANTERIOR'] == 0)],
}

linhas = []
for nome, sub in cortes.items():
    r = sub.groupby('DET_ANTERIOR')['DETRATOR'].mean() * 100
    linhas.append({'corte': nome, 'n': len(sub),
                   'sem historico (%)': round(r[0.0], 1),
                   'com historico (%)': round(r[1.0], 1),
                   'razao': round(r[1.0] / r[0.0], 1)})

resultado = pd.DataFrame(linhas).set_index('corte')
print(resultado)

## 6. Visualização

O gráfico mostra o que sustenta a hipótese: conforme as causas operacionais são removidas, a
distância entre os dois grupos **aumenta** em vez de diminuir.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(resultado))
largura = 0.38

ax.bar(x - largura/2, resultado['sem historico (%)'], largura,
       label='Sem detracao anterior', color='#7fb3d5')
ax.bar(x + largura/2, resultado['com historico (%)'], largura,
       label='Com detracao anterior', color='#0a4d8c')

for i, (_, r) in enumerate(resultado.iterrows()):
    ax.text(i, r['com historico (%)'] + 1.5, f"{r['razao']}x",
            ha='center', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([c.split('. ')[1] for c in resultado.index], fontsize=9)
ax.set_ylabel('Taxa de detracao (%)')
ax.set_title('Efeito do historico de detracao sob controles progressivos')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=.25)
fig.tight_layout()
plt.show()

## 7. Modelo ajustado

A regressão logística estima o efeito do histórico com as condições operacionais do voo
incluídas como controle, de modo que o resultado não possa ser atribuído a elas.

In [ ]:
# CANCELADO ja e o indicador booleano normalizado na secao 1. Usa-lo direto evita o
# get_dummies, que devolveria a coluna intacta, sem virar indicador, se o tipo mudasse.
X = pd.DataFrame({
    'CANCELADO':      hist['CANCELADO'].astype(float).values,
    'ATRASO_CHEGADA': hist['ATRASO_CHEGADA'].astype(float).values,
    'DET_ANTERIOR':   hist['DET_ANTERIOR'].astype(float).values,
})
X = sm.add_constant(X)

modelo = sm.Logit(hist['DETRATOR'].astype(float).values, X).fit(disp=0)

odds = np.exp(modelo.params['DET_ANTERIOR'])
ic   = np.exp(modelo.conf_int().loc['DET_ANTERIOR'])

print(f'n = {len(hist)}')
print(f'odds ratio do historico de detracao = {odds:.2f}')
print(f'IC 95% = [{ic[0]:.2f}, {ic[1]:.2f}]')
print(f'p = {modelo.pvalues["DET_ANTERIOR"]:.3e}')

## Conclusão

- **Concentração**: a combinação "detratou nas duas" aparece muito acima do esperado sob
  independência, com qui-quadrado alto e p < 0,001. A repetição não é acaso.
- **Predição**: o histórico separa a base em dois grupos com risco cerca de três vezes
  diferente.
- **Resistência ao controle**: a razão entre os grupos **aumenta** conforme as causas
  operacionais são removidas. É o oposto do que se esperaria se o efeito fosse do voo, e é o
  resultado que sustenta a hipótese.
- **Modelo ajustado**: o histórico permanece fortemente associado à detração mesmo controlando
  por atraso na chegada e cancelamento.

**Para o modelo preditivo.** O histórico de detração é um preditor forte e legítimo, já que a
resposta anterior existe antes do voo novo e portanto não gera vazamento. É também um alerta de
viés, porque parte do que hoje se atribui ao atraso pode ser o mesmo Cliente insatisfeito
aparecendo repetidas vezes.

**Ressalvas.** O histórico existe para apenas cerca de 16% da base, o que torna a variável um
preditor complementar e nunca principal. Além disso, os dados não permitem separar insatisfação
crônica de estilo de resposta, ou seja, a tendência de certas pessoas a usarem sempre o extremo
baixo da escala. Por isso o padrão é tratado aqui como indício consistente de um traço de
detrator crônico, e não como confirmação definitiva de sua existência.